In [ ]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('..')
import os

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets

from src.dataset import get_transforms, make_loaders
from src.evaluate import get_logits, predict_loader, summarize
from src.experiment import run_experiment, run_experiment_tl
from src.model import CNNBaseline, TLResNet, describe_run
from src.train import fit, make_scheduler
from src.utils import get_device, set_seed

SEED, EPOCHS, IMG_SIZE, BATCH = 42, 30, 128, 32
DATA = '../data/processed'
RESULTS = '../results/ablations.csv'
device = get_device()
paths = {
    'data': '../data/processed',
    'ckpt_dir': '../checkpoints',
    'log_dir': '../logs',
    'results': '../results/experiments.csv',
}

In [ ]:
img_size = 224
_, eval_tf = get_transforms(img_size, augment=False)
train_ds = datasets.ImageFolder('../data/processed/train', transform=eval_tf)
val_ds   = datasets.ImageFolder('../data/processed/val', transform=eval_tf)
g = torch.Generator(); g.manual_seed(SEED)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, generator=g)
val_loader   = DataLoader(val_ds, batch_size=32, shuffle=False)

In [ ]:
set_seed(SEED)

model = TLResNet(trainable=('fc',)).to(device)
opt = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

res_fe = run_experiment_tl(
    name='resnet_fe',
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=opt,
    criterion=nn.CrossEntropyLoss(),
    device=device,
    epochs=20,
    extra={'lr_head': 1e-3, 'trainable': 'fc', 'img_size': 224},
)